# MinerU в Google Colab (пошагово)

Сравнение вывода **MinerU** с эталонами рядом с PNG (`*.ref.txt`, `*.ref.md`, …). Запускается скрипт репозитория **`scripts/mineru_image_benchmark.py`**.

## Порядок (выполняйте ячейки сверху вниз)

1. **Runtime → Change runtime type → GPU** (рекомендуется; иначе дольше и может не хватить памяти).
2. **Шаг 2** — клон репозитория + `pip install jiwer`.
3. **Шаг 3** — установка **MinerU** (`mineru[pipeline]` или `mineru[all]`).
4. **Шаг 4** — проверка путей, список PNG и эталонов.
5. **Шаг 5 (опционально)** — если модели с Hugging Face не качаются: `USE_MODELSCOPE_FOR_MINERU=1` и кодовая ячейка шага 5.
6. **Шаг 6** — прогон бенчмарка (вызов `mineru` по каждому изображению).

**Результаты:** `output/mineru_benchmark/hypotheses/mineru/*.md`, `mineru_runs.jsonl`, `mineru_summaries.json`.

**Другой репозиторий / форк:** переменные **`OCR_ANALYZE_GIT_URL`**, **`OCR_ANALYZE_COLAB_DIR`** (как в ноутбуке GOT).

После **первой** установки MinerU в Colab часто нужен **Runtime → Restart session**, затем снова шаги **2–4** и шаг **6**.


In [ ]:
# Шаг 2 — Colab: клон репозитория + jiwer (нужен скрипту метрик).

from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


GIT_URL = os.environ.get(
    "OCR_ANALYZE_GIT_URL",
    "https://github.com/developer-mixa/OCR-Analyze.git",
)
REPO_DIR = Path(os.environ.get("OCR_ANALYZE_COLAB_DIR", "/content/OCR-Analyze"))

if in_colab():
    if not (REPO_DIR / "scripts").is_dir():
        print("Клонирую", GIT_URL, "→", REPO_DIR)
        subprocess.check_call(["git", "clone", "--depth", "1", GIT_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    print("Рабочий каталог:", Path.cwd().resolve())
else:
    print("Не Colab — клон не выполняется. cwd:", Path.cwd().resolve())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "jiwer"])
print("OK: jiwer установлен.")
print("Дальше: шаг 3 (MinerU).")


In [ ]:
# Шаг 3 — установка MinerU (долго, много места на диске).
# По умолчанию: mineru[pipeline] (легче). Для полного набора: import os; os.environ["MINERU_PKG"]="all"

import os
import shutil
import subprocess
import sys
from pathlib import Path

PKG = os.environ.get("MINERU_PKG", "pipeline").strip().lower()  # pipeline | all
if PKG not in ("pipeline", "all"):
    raise ValueError("MINERU_PKG должен быть pipeline или all")

spec = "mineru[pipeline]" if PKG == "pipeline" else "mineru[all]"
print("Устанавливаю", spec, "...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", spec])
print("which(mineru) =", shutil.which("mineru"))
print("Если None — Runtime → Restart session, затем снова шаги 2–3.")


In [ ]:
# Шаг 4 — корень репозитория, входные данные.

from __future__ import annotations

from pathlib import Path

REPO_ROOT_OVERRIDE: Path | None = None


def find_repo_root() -> Path:
    if REPO_ROOT_OVERRIDE is not None:
        p = REPO_ROOT_OVERRIDE.expanduser().resolve()
        if (p / "scripts").is_dir():
            return p
    cwd = Path.cwd().resolve()
    for start in [cwd, *cwd.parents]:
        if (start / "scripts" / "mineru_image_benchmark.py").is_file():
            return start
    return cwd


REPO_ROOT = find_repo_root()
INPUT_DIR = REPO_ROOT / "input" / "data" / "1"
SCRIPT = REPO_ROOT / "scripts" / "mineru_image_benchmark.py"
OUT_DIR = REPO_ROOT / "output" / "mineru_benchmark"

print("REPO_ROOT =", REPO_ROOT.resolve())
print("SCRIPT exists:", SCRIPT.is_file(), SCRIPT)
print("INPUT_DIR =", INPUT_DIR.resolve(), "exists:", INPUT_DIR.is_dir())

_png = sorted(INPUT_DIR.glob("*.png")) if INPUT_DIR.is_dir() else []
print(f"PNG: {len(_png)}")
for p in _png:
    stem = p.stem
    refs = [n for n in (f"{stem}.ref.txt", f"{stem}.ref.md", f"{stem}.txt", f"{stem}.md") if (INPUT_DIR / n).is_file()]
    print(" ", p.name, "| эталон:", ", ".join(refs) if refs else "нет")
if not _png:
    print("Добавьте PNG в input/data/1 (в клоне репозитория).")


## Шаг 5 (опционально) — зеркало моделей

Если загрузка весов с Hugging Face из Colab падает или очень медленная, задайте **`USE_MODELSCOPE_FOR_MINERU=1`** (в ячейке выше: `import os; os.environ["USE_MODELSCOPE_FOR_MINERU"]="1"`) и выполните **следующую кодовую ячейку**. Подробнее: [Quick Usage — MinerU](https://opendatalab.github.io/MinerU/usage/quick_usage/).


In [ ]:
# Шаг 5 — ModelScope вместо Hugging Face (только если USE_MODELSCOPE_FOR_MINERU=1).

import os

if os.environ.get("USE_MODELSCOPE_FOR_MINERU", "").strip() == "1":
    os.environ["MINERU_MODEL_SOURCE"] = "modelscope"
    print("MINERU_MODEL_SOURCE =", os.environ["MINERU_MODEL_SOURCE"])
else:
    print("Пропуск: задайте USE_MODELSCOPE_FOR_MINERU=1 перед этой ячейкой, если нужен ModelScope.")


## Шаг 6 — прогон `mineru_image_benchmark.py`

Ниже вызывается скрипт: для каждого PNG — `mineru` → markdown → CER к эталону. Параметры можно менять в `cmd` или через переменные окружения (`MINERU_BACKEND`, `MINERU_LANG`, `MINERU_METHOD`).


In [ ]:
# Шаг 6 — запуск бенчмарка MinerU.

import os
import shutil
import subprocess
import sys
from pathlib import Path

if "REPO_ROOT" not in globals() or "SCRIPT" not in globals():
    raise RuntimeError("Сначала выполните шаг 4.")

if not shutil.which("mineru"):
    raise RuntimeError("mineru не в PATH. Выполните шаг 3 (и при необходимости Restart runtime).")

cmd = [
    sys.executable,
    str(SCRIPT),
    "--input-dir",
    str(INPUT_DIR),
    "--output-dir",
    str(OUT_DIR),
    "--backend",
    os.environ.get("MINERU_BACKEND", "pipeline"),
    "--lang",
    os.environ.get("MINERU_LANG", "cyrillic"),
]
if os.environ.get("MINERU_METHOD"):
    cmd.extend(["--method", os.environ["MINERU_METHOD"]])

print("Команда:", " ".join(cmd))
subprocess.check_call(cmd, cwd=str(REPO_ROOT))

summ = OUT_DIR / "mineru_summaries.json"
if summ.is_file():
    print("\n---", summ, "---\n")
    print(summ.read_text(encoding="utf-8"))
